# 硬提示

### 以下部分与软提示训练相同

In [ ]:
!pip install mindnlp==0.4.0
!pip uninstall mindformers -y
!pip install transformers==4.40.0
!pip install mindspore==2.4.1
%env HF_ENDPOINT=https://hf-mirror.com

# 包含软提示所需的依赖库以及处理数据集的库
import os

import mindspore
from mindnlp.core.optim import AdamW
from tqdm import tqdm
import evaluate
from mindnlp.dataset import load_dataset
from mindnlp.engine import set_seed
from mindnlp.transformers import AutoModelForSequenceClassification, AutoTokenizer
from mindnlp.transformers.optimization import get_linear_schedule_with_warmup
from mindnlp.dataset import BaseMapFunction

import numpy as np
import math
import random
import codecs
from pathlib import Path

import mindspore.dataset as ds
from mindspore import Tensor
from mindspore import context
from mindspore.train.model import Model
from mindspore.nn.metrics import Accuracy
from mindspore.train.serialization import load_checkpoint, load_param_into_net
from mindspore.train.callback import ModelCheckpoint, CheckpointConfig, LossMonitor, TimeMonitor
from mindspore.ops import operations as ops

from mindnlp.peft import (
    get_peft_config,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
    PeftType,
    PromptTuningConfig,
)
peft_type = PeftType.PROMPT_TUNING
peft_config = PromptTuningConfig(task_type="SEQ_CLS", num_virtual_tokens=10)
from easydict import EasyDict as edict
cfg = edict({
    'model_name': "AI-ModelScope/roberta-large",
    'batch_size': 64,
    'epoch_size': 10,
    'data_path': './data/',
    'device_target': 'Ascend',
    'lr':1e-3,
    'task': "mrpc"
})

mindspore.set_context(device_target=cfg.device_target)

## 新增部分：定义hard_prompt提示部分

In [ ]:
hard_prompt = {
    "positivePrompt": [
        "This review expresses a positive sentiment. The review is: "
    ],
    "negativePrompt": [
        "This review expresses a negative sentiment. The review is: "
    ]
}

### 以下依然与软提示训练相同

In [ ]:
# load tokenizer
if any(k in cfg.model_name for k in ("gpt", "opt", "bloom")):
    padding_side = "left"
else:
    padding_side = "right"

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, padding_side=padding_side, mirror="modelscope")
if getattr(tokenizer, "pad_token_id") is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class Generator():
    def __init__(self, input_list):
        self.input_list=input_list
    def __getitem__(self,item):
        # 这里的item是数据集的索引
        # 这里的self.input_list是一个list，里面的每个元素都是一个list
        # 这里的self.input_list[item]是一个list，里面的第一个元素是数据，第二个元素是标签
        # 这里的self.input_list[item][0]是list数据，self.input_list[item][1]是int标签
        return (self.input_list[item][0],
                np.array(self.input_list[item][1],dtype=np.int32),item) # 返回数据与标签
    def __len__(self):
        return len(self.input_list)

## 修改MovieReview类的`read_data`函数

In [ ]:
class MovieReview:
    '''
    影评数据集
    '''
    def __init__(self, root_dir, split):
        '''
        input:
            root_dir: 影评数据目录
            maxlen: 设置句子最大长度
            split: 设置数据集中训练/评估的比例
        '''
        self.path = root_dir
        self.feelMap = {
            'neg':0,
            'pos':1
        }
        self.files = []

        self.doConvert = False
        
        mypath = Path(self.path)
        if not mypath.exists() or not mypath.is_dir():
            print("please check the root_dir!")
            raise ValueError

        # 在数据目录中找到文件
        for root,_,filename in os.walk(self.path):
            for each in filename:
                self.files.append(os.path.join(root,each))
            break

        # 确认是否为两个文件.neg与.pos
        if len(self.files) != 2:
            print("There are {} files in the root_dir".format(len(self.files)))
            raise ValueError

        # 读取数据
        self.word_num = 0
        self.maxlen = 0
        self.minlen = float("inf")
        self.maxlen = float("-inf")
        self.Pos = []
        self.Neg = []
        for filename in self.files:
            f = codecs.open(filename, 'r')
            ff = f.read()
            file_object = codecs.open(filename, 'w', 'utf-8')
            file_object.write(ff)
            self.read_data(filename)
        self.PosNeg = self.Pos + self.Neg
        self.split_dataset(split=split)

    def read_data(self, filePath):

        with open(filePath,'r') as f:
            
            for sentence in f.readlines():
                if filePath.endswith('.pos'):
                    # 影评为正面
                    sentence = hard_prompt['positivePrompt'][0] + " ," + sentence
                else :
                    # 影评为负面
                    sentence = hard_prompt['negativePrompt'][0] + " ," + sentence
                sentence = sentence.replace('\n','')\
                                    .replace('"','')\
                                    .replace('\'','')\
                                    .replace('.','')\
                                    .replace(',','')\
                                    .replace('[','')\
                                    .replace(']','')\
                                    .replace('(','')\
                                    .replace(')','')\
                                    .replace(':','')\
                                    .replace('--','')\
                                    .replace('-',' ')\
                                    .replace('\\','')\
                                    .replace('0','')\
                                    .replace('1','')\
                                    .replace('2','')\
                                    .replace('3','')\
                                    .replace('4','')\
                                    .replace('5','')\
                                    .replace('6','')\
                                    .replace('7','')\
                                    .replace('8','')\
                                    .replace('9','')\
                                    .replace('`','')\
                                    .replace('=','')\
                                    .replace('$','')\
                                    .replace('/','')\
                                    .replace('*','')\
                                    .replace(';','')\
                                    .replace('<b>','')\
                                    .replace('%','')
                if sentence:
                    self.word_num += len(sentence)
                    self.maxlen = self.maxlen if self.maxlen >= len(sentence) else len(sentence)
                    self.minlen = self.minlen if self.minlen <= len(sentence) else len(sentence)
                    if 'pos' in filePath:
                        self.Pos.append([sentence,self.feelMap['pos']])
                    else:
                        self.Neg.append([sentence,self.feelMap['neg']])

    def text2vec(self, maxlen):
        '''
        将句子转化为向量

        '''
        # Vocab = {word : index}
        self.Vocab = dict()

        # self.Vocab['None']
        for SentenceLabel in self.Pos+self.Neg:
            vector = [0]*maxlen
            for index, word in enumerate(SentenceLabel[0]):
                if index >= maxlen:
                    break
                if word not in self.Vocab.keys():
                    self.Vocab[word] = len(self.Vocab)
                    vector[index] = len(self.Vocab) - 1
                else:
                    vector[index] = self.Vocab[word]
            SentenceLabel[0] = vector
        self.doConvert = True

    def split_dataset(self, split):
        '''
        分割为训练集与测试集

        '''

        trunk_pos_size = math.ceil((1-split)*len(self.Pos))
        trunk_neg_size = math.ceil((1-split)*len(self.Neg))
        trunk_num = int(1/(1-split))
        pos_temp=list()
        neg_temp=list()
        for index in range(trunk_num):
            pos_temp.append(self.Pos[index*trunk_pos_size:(index+1)*trunk_pos_size])
            neg_temp.append(self.Neg[index*trunk_neg_size:(index+1)*trunk_neg_size])
        self.test = pos_temp.pop(2)+neg_temp.pop(2)
        self.train = [i for item in pos_temp+neg_temp for i in item]

        random.shuffle(self.train)
        random.shuffle(self.test)

    def get_dict_len(self):
        '''
        获得数据集中文字组成的词典长度
        '''
        if self.doConvert:
            return len(self.Vocab)
        else:
            print("Haven't finished Text2Vec")
            return -1

    def create_train_dataset(self):
        dataset = ds.GeneratorDataset(source=Generator(input_list=self.train), column_names=["data","label","idx"], shuffle=False)
        return dataset

    def create_test_dataset(self):
        dataset = ds.GeneratorDataset(source=Generator(input_list=self.test), column_names=["data","label","idx"], shuffle=False)
        return dataset

### 其余部分保持不变

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name, return_dict=True,mirror="modelscope")

class MapFunc(BaseMapFunction):
    def __call__(self, data , label,idx):
        outputs = tokenizer(data ,truncation=True, max_length=None)
        return outputs['input_ids'], outputs['attention_mask'], label


def get_dataset(dataset, tokenizer):
    input_colums=['data', 'label', 'idx']
    output_columns=['input_ids', 'attention_mask', 'labels']
    dataset = dataset.map(MapFunc(input_colums, output_columns),
                          input_colums, output_columns)
    dataset = dataset.padded_batch(cfg.batch_size , pad_info={'input_ids': (None, tokenizer.pad_token_id),
                                                         'attention_mask': (None, 0)})
    return dataset
instance = MovieReview(root_dir=cfg.data_path , split=0.9)
train_dataset = instance.create_train_dataset()
test_dataset = instance.create_test_dataset()
train_dataset = get_dataset(train_dataset, tokenizer)
test_dataset = get_dataset(test_dataset, tokenizer)

optimizer = AdamW(params=model.trainable_params(), lr=cfg.lr)

lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0.06 * (len(train_dataset) * cfg.epoch_size),
    num_training_steps=(len(train_dataset) * cfg.epoch_size)
)

metric = evaluate.load("glue", cfg.task) # glue数据集

def forward_fn(**batch):
    outputs = model(**batch)
    loss = outputs.loss
    return loss

grad_fn = mindspore.value_and_grad(forward_fn, None, model.trainable_params())

for epoch in range(cfg.epoch_size):
    model.set_train()
    train_total_size = train_dataset.get_dataset_size()
    for step, batch in enumerate(tqdm(train_dataset.create_dict_iterator(), total=train_total_size)):

        loss, grads = grad_fn(**batch)
        optimizer.step(grads)
        lr_scheduler.step()

    model.set_train(False)
    eval_total_size = test_dataset.get_dataset_size()
    for step, batch in enumerate(tqdm(test_dataset.create_dict_iterator(), total=eval_total_size)):
        outputs = model(**batch)
        predictions = outputs.logits.argmax(axis=-1)
        predictions, references = predictions, batch["labels"]
        metric.add_batch(
            predictions=predictions,
            references=references,
        )

    eval_metric = metric.compute()
    print(f"epoch {epoch}:", eval_metric)